In [1]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import statsmodels.api as sm

import concurrent.futures


import numpy as np
import pandas as pd
import ast
import glob
import pickle
import dask
import os
import itertools

import pickle

#from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances, mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from statsmodels.regression.rolling import RollingOLS

from tqdm.notebook import tqdm

from multiprocessing import Pool, cpu_count
from joblib import Parallel, delayed
#from tqdm import tqdm
from collections import Counter
from functools import reduce


import dask
import dask.dataframe as dd
from dask.distributed import Client
from dask.diagnostics import ProgressBar

#client = Client(n_workers=20, memory_limit="10GB", interface='lo')
from concurrent.futures import ThreadPoolExecutor

import dask_ml.cluster as dask_cluster

from pprint import pprint
import os

pd.set_option('display.max_columns', None)

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

### Load Augmented DF

In [2]:
augmented_df = dd.read_csv("../../data/augmented_us-counties_latest.csv", assume_missing=True).compute()
augmented_df["date"] = pd.to_datetime(augmented_df["date"])
augmented_df["fips"] = augmented_df["fips"].astype(int)
augmented_df["days_from_start"] = augmented_df["days_from_start"].astype(int)
augmented_df["log_rolled_cases"] = np.log(augmented_df["rolled_cases"] + 1.1)
augmented_df = augmented_df.sort_values(by=["fips","date"])

# Generate 7 day validation data
for shift in range(1,8):
    augmented_df["shifted_log_rolled_cases_{}".format(shift)] = augmented_df.groupby("fips")["log_rolled_cases"].shift(-shift)

# Check for gaps
gt_columns = ["fips", "days_from_start", "date", "log_rolled_cases"] + ["shifted_log_rolled_cases_{}".format(shift) for shift in range(1,8)]
augmented_df_gt = augmented_df[gt_columns]
grouped = augmented_df_gt.groupby('fips')

for fips, group in grouped:
    missing_days = group['days_from_start'].diff().gt(1).sum()
    if missing_days > 0:
        print(f"Gap(s) found in 'days_from_start' for fips {fips}: {missing_days} gap(s)")


#df = augmented_df.copy()
window_sizes = list(range(2,15))
fips_list = augmented_df_gt["fips"].unique()

### Load all $r_{t,c}$

In [3]:
all_beta_df = dd.read_csv("../benchmark_fixed_window/Fixed_windows_all_beta.csv",assume_missing=True).compute()
all_beta_df["date"] = pd.to_datetime(all_beta_df["date"])
all_beta_df = all_beta_df.sort_values(by=["fips","date"])
(all_beta_df.head())

,fips,days_from_start,date,log_rolled_cases,shifted_log_rolled_cases,beta_wsize=2,beta_wsize=3,beta_wsize=4,beta_wsize=5,beta_wsize=6,beta_wsize=7,beta_wsize=8,beta_wsize=9,beta_wsize=10,beta_wsize=11,beta_wsize=12,beta_wsize=13,beta_wsize=14
0,1001.0,69.0,2020-03-30,1.831438,2.469309,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1001.0,70.0,2020-03-31,1.960095,2.528012,0.128657,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1001.0,71.0,2020-04-01,2.074070,2.550561,0.113975,0.121316,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1001.0,72.0,2020-04-02,2.143422,2.625703,0.069352,0.091664,0.104993,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1001.0,73.0,2020-04-03,2.239189,2.676117,0.095767,0.082559,0.090663,0.099883,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Add Predictions

In [4]:
all_beta_df_results = all_beta_df.copy()
window_sizes = range(2,15)
for window_size in window_sizes:
    for shift in range(1,8):
        all_beta_df_results["predicted_beta_wsize={}_shift={}".format(window_size, shift)] = all_beta_df_results["beta_wsize={}".format(window_size)]*shift + all_beta_df_results["log_rolled_cases"]
all_beta_df_results = all_beta_df_results.drop(columns=["date","log_rolled_cases", "shifted_log_rolled_cases"])
all_beta_df_results = pd.merge(augmented_df_gt, all_beta_df_results, on=["fips", "days_from_start"], how="left")
all_beta_df_results

,fips,days_from_start,date,log_rolled_cases,shifted_log_rolled_cases_1,shifted_log_rolled_cases_2,shifted_log_rolled_cases_3,shifted_log_rolled_cases_4,shifted_log_rolled_cases_5,shifted_log_rolled_cases_6,shifted_log_rolled_cases_7,beta_wsize=2,beta_wsize=3,beta_wsize=4,beta_wsize=5,beta_wsize=6,beta_wsize=7,beta_wsize=8,beta_wsize=9,beta_wsize=10,beta_wsize=11,beta_wsize=12,beta_wsize=13,beta_wsize=14,predicted_beta_wsize=2_shift=1,predicted_beta_wsize=2_shift=2,predicted_beta_wsize=2_shift=3,predicted_beta_wsize=2_shift=4,predicted_beta_wsize=2_shift=5,predicted_beta_wsize=2_shift=6,predicted_beta_wsize=2_shift=7,predicted_beta_wsize=3_shift=1,predicted_beta_wsize=3_shift=2,predicted_beta_wsize=3_shift=3,predicted_beta_wsize=3_shift=4,predicted_beta_wsize=3_shift=5,predicted_beta_wsize=3_shift=6,predicted_beta_wsize=3_shift=7,predicted_beta_wsize=4_shift=1,predicted_beta_wsize=4_shift=2,predicted_beta_wsize=4_shift=3,predicted_beta_wsize=4_shift=4,predicted_beta_wsize=4_shift=5,predicted_beta_wsize=4_shift=6,predicted_beta_wsize=4_shift=7,predicted_beta_wsize=5_shift=1,predicted_beta_wsize=5_shift=2,predicted_beta_wsize=5_shift=3,predicted_beta_wsize=5_shift=4,predicted_beta_wsize=5_shift=5,predicted_beta_wsize=5_shift=6,predicted_beta_wsize=5_shift=7,predicted_beta_wsize=6_shift=1,predicted_beta_wsize=6_shift=2,predicted_beta_wsize=6_shift=3,predicted_beta_wsize=6_shift=4,predicted_beta_wsize=6_shift=5,predicted_beta_wsize=6_shift=6,predicted_beta_wsize=6_shift=7,predicted_beta_wsize=7_shift=1,predicted_beta_wsize=7_shift=2,predicted_beta_wsize=7_shift=3,predicted_beta_wsize=7_shift=4,predicted_beta_wsize=7_shift=5,predicted_beta_wsize=7_shift=6,predicted_beta_wsize=7_shift=7,predicted_beta_wsize=8_shift=1,predicted_beta_wsize=8_shift=2,predicted_beta_wsize=8_shift=3,predicted_beta_wsize=8_shift=4,predicted_beta_wsize=8_shift=5,predicted_beta_wsize=8_shift=6,predicted_beta_wsize=8_shift=7,predicted_beta_wsize=9_shift=1,predicted_beta_wsize=9_shift=2,predicted_beta_wsize=9_shift=3,predicted_beta_wsize=9_shift=4,predicted_beta_wsize=9_shift=5,predicted_beta_wsize=9_shift=6,predicted_beta_wsize=9_shift=7,predicted_beta_wsize=10_shift=1,predicted_beta_wsize=10_shift=2,predicted_beta_wsize=10_shift=3,predicted_beta_wsize=10_shift=4,predicted_beta_wsize=10_shift=5,predicted_beta_wsize=10_shift=6,predicted_beta_wsize=10_shift=7,predicted_beta_wsize=11_shift=1,predicted_beta_wsize=11_shift=2,predicted_beta_wsize=11_shift=3,predicted_beta_wsize=11_shift=4,predicted_beta_wsize=11_shift=5,predicted_beta_wsize=11_shift=6,predicted_beta_wsize=11_shift=7,predicted_beta_wsize=12_shift=1,predicted_beta_wsize=12_shift=2,predicted_beta_wsize=12_shift=3,predicted_beta_wsize=12_shift=4,predicted_beta_wsize=12_shift=5,predicted_beta_wsize=12_shift=6,predicted_beta_wsize=12_shift=7,predicted_beta_wsize=13_shift=1,predicted_beta_wsize=13_shift=2,predicted_beta_wsize=13_shift=3,predicted_beta_wsize=13_shift=4,predicted_beta_wsize=13_shift=5,predicted_beta_wsize=13_shift=6,predicted_beta_wsize=13_shift=7,predicted_beta_wsize=14_shift=1,predicted_beta_wsize=14_shift=2,predicted_beta_wsize=14_shift=3,predicted_beta_wsize=14_shift=4,predicted_beta_wsize=14_shift=5,predicted_beta_wsize=14_shift=6,predicted_beta_wsize=14_shift=7
0,1001,69,2020-03-30,1.831438,1.960095,2.074070,2.143422,2.239189,2.326581,2.406945,2.469309,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1001,70,2020-03-31,1.960095,2.074070,2.143422,2.239189,2.326581,2.406945,2.469309,2.528012,0.128657,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.088752,2.217408,2.346065,2.474722,2.603379,2.732036,2.860693,NaN,NaN,NaN,NaN

In [24]:
all_beta_df_results.columns

Index(['fips', 'days_from_start', 'date', 'log_rolled_cases',
       'shifted_log_rolled_cases_1', 'shifted_log_rolled_cases_2',
       'shifted_log_rolled_cases_3', 'shifted_log_rolled_cases_4',
       'shifted_log_rolled_cases_5', 'shifted_log_rolled_cases_6',
       ...
       'diff_wsize=13_shift=5', 'diff_wsize=13_shift=6',
       'diff_wsize=13_shift=7', 'diff_wsize=14_shift=1',
       'diff_wsize=14_shift=2', 'diff_wsize=14_shift=3',
       'diff_wsize=14_shift=4', 'diff_wsize=14_shift=5',
       'diff_wsize=14_shift=6', 'diff_wsize=14_shift=7'],
      dtype='object', length=206)

### Save the Shifts Beta

In [ ]:
wsize_shift = itertools.product(list(range(2,15)), list(range(1,8)))
cols_to_keep = ["fips", "days_from_start", "date"] + ["diff_wsize={}_shift={}".format(window_size, shift) for window_size, shift in wsize_shift]
all_beta_df_results_diff = all_beta_df_results[cols_to_keep]
all_beta_df_results_diff
all_beta_df_results_diff.to_csv("Fixed_Windows_Validation_Diff.csv", index=False)